# Traitement TRI EER Agence — fusion recto/verso

Pipeline :
1. On repère les fichiers `id_recto_xxx.*` / `id_verso_xxx.*`
2. Conversion des images (jpeg/png) en PDF
3. Décision :
   - recto déjà à 2 pages (ou plus) → on garde tel quel, verso ignoré
   - recto à 1 page + verso présent → concaténation
   - recto à 1 page sans verso → on garde tel quel (anomalie loguée)
   - verso sans recto → anomalie loguée, on garde le verso seul
4. Passe de contrôle : sur les documents à 2 pages, on hash chaque page (phash) pour détecter
   les cas où recto+verso étaient en fait sur la même page scannée deux fois → on supprime le doublon
5. Renommage final `TRI_CNI_ETR_0001.pdf`, `TRI_CNI_ETR_0002.pdf`, ... + log + CSV récapitulatif

Dépendances : `pip install pymupdf imagehash pillow pandas`


In [ ]:
import re, io, logging
from pathlib import Path
from collections import defaultdict

import fitz  # PyMuPDF
import imagehash
from PIL import Image
import pandas as pd

# --- config à adapter ---
INPUT_DIR = Path("classifier_eer/TRI/OK EER Agence")
OUTPUT_DIR = Path("output_tri")
OUTPUT_DIR.mkdir(exist_ok=True)
PREFIX = "TRI_CNI_ETR"
HASH_THRESHOLD = 5  # distance de Hamming max pour considerer 2 pages identiques

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(OUTPUT_DIR / "traitement.log", mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("tri_eer")


## 1. Repérage et regroupement des fichiers par id

In [ ]:
FILENAME_RE = re.compile(r"^(?P<id>.+?)_(?P<side>recto|verso)_[^.]+\.(?P<ext>pdf|jpe?g|png)$", re.IGNORECASE)

groups = defaultdict(dict)  # id -> {"recto": path, "verso": path}

for f in sorted(INPUT_DIR.iterdir()):
    if not f.is_file():
        continue
    m = FILENAME_RE.match(f.name)
    if not m:
        log.warning(f"Fichier ignore (pattern non reconnu): {f.name}")
        continue
    doc_id, side = m.group("id"), m.group("side").lower()
    if side in groups[doc_id]:
        log.warning(f"{doc_id}: plusieurs fichiers '{side}' trouves, {groups[doc_id][side].name} garde, {f.name} ignore")
        continue
    groups[doc_id][side] = f

log.info(f"{len(groups)} documents identifies")


## 2. Fonctions utilitaires : conversion en PDF + hash de page

In [ ]:
def to_pdf_doc(path: Path) -> fitz.Document:
    """Ouvre un fichier (pdf ou image) et renvoie un fitz.Document en pdf."""
    if path.suffix.lower() == ".pdf":
        return fitz.open(path)
    img_doc = fitz.open(path)
    pdf_bytes = img_doc.convert_to_pdf()
    img_doc.close()
    return fitz.open("pdf", pdf_bytes)


def page_hash(doc: fitz.Document, page_index: int) -> imagehash.ImageHash:
    pix = doc[page_index].get_pixmap(dpi=100)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return imagehash.phash(img)


## 3. Décision recto/verso + concaténation

In [ ]:
records = []  # recap pour le log final

for doc_id, sides in groups.items():
    recto_path = sides.get("recto")
    verso_path = sides.get("verso")

    if recto_path is None and verso_path is None:
        continue

    if recto_path is None:
        merged = to_pdf_doc(verso_path)
        status = "verso seul (recto manquant)"
        log.warning(f"{doc_id}: {status}")
        records.append({"id": doc_id, "doc": merged, "status": status})
        continue

    recto_doc = to_pdf_doc(recto_path)

    if recto_doc.page_count >= 2:
        merged = recto_doc
        status = f"recto deja {recto_doc.page_count} pages -> verso non ajoute"
        if verso_path is not None:
            status += " (verso present mais ignore)"
        log.info(f"{doc_id}: {status}")
    elif verso_path is not None:
        verso_doc = to_pdf_doc(verso_path)
        merged = fitz.open()
        merged.insert_pdf(recto_doc)
        merged.insert_pdf(verso_doc)
        verso_doc.close()
        recto_doc.close()
        status = "recto (1 page) + verso concatenes"
        log.info(f"{doc_id}: {status}")
    else:
        merged = recto_doc
        status = "recto seul (verso manquant)"
        log.warning(f"{doc_id}: {status}")

    records.append({"id": doc_id, "doc": merged, "status": status})


## 4. Passe de contrôle par hashing

Sur les documents à 2 pages, on vérifie si les deux pages sont en fait identiques
(cas où recto+verso etaient deja sur le meme scan par ex.). Si la distance de Hamming
entre les deux phash est faible, on supprime la 2e page.

In [ ]:
for rec in records:
    doc = rec["doc"]
    if doc.page_count != 2:
        continue
    h0, h1 = page_hash(doc, 0), page_hash(doc, 1)
    dist = h0 - h1
    if dist <= HASH_THRESHOLD:
        doc.delete_page(1)
        rec["status"] += f" | pages identiques detectees (hamming={dist}) -> page 2 supprimee"
        log.info(f"{rec['id']}: pages 1 et 2 quasi-identiques (distance={dist}), page 2 supprimee")
    else:
        rec["status"] += f" | pages differentes (hamming={dist}), conservees"


## 5. Écriture finale + renommage + récapitulatif

In [ ]:
for i, rec in enumerate(records, start=1):
    out_name = f"{PREFIX}_{i:04d}.pdf"
    out_path = OUTPUT_DIR / out_name
    rec["doc"].save(out_path)
    rec["doc"].close()
    rec["output_file"] = out_name
    log.info(f"{rec['id']} -> {out_name} ({rec['status']})")

df = pd.DataFrame([{k: v for k, v in r.items() if k != "doc"} for r in records])
df.to_csv(OUTPUT_DIR / "recap_traitement.csv", index=False)
df
